# 🔍 Data Structure Diagnostic Tool

**Purpose:** Identify why alternative roles and crosswalk data aren't merging properly.

**Instructions:**
1. Upload the same 4 files you used before
2. Run all cells
3. Review the detailed diagnostic output
4. Share results so we can create a fixed version


In [ ]:
# Install dependencies
!pip -q install pandas numpy


In [ ]:
# Upload files
from google.colab import files
import os, zipfile, glob, json
import pandas as pd
import numpy as np

uploaded = files.upload()

ZIP_NAME = "Likelihood Evaluation Resources.zip"
WORKDIR = "/content/likelihood_eval"
os.makedirs(WORKDIR, exist_ok=True)

if ZIP_NAME in uploaded:
    zip_path = os.path.join("/content", ZIP_NAME)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(WORKDIR)
    print(f"✅ Extracted {ZIP_NAME}")

for fn in uploaded.keys():
    src = os.path.join("/content", fn)
    dst = os.path.join(WORKDIR, fn)
    if os.path.exists(src) and src != dst:
        os.replace(src, dst)

print(f"\n📁 Files in {WORKDIR}:")
for f in glob.glob(os.path.join(WORKDIR, '*')):
    size_kb = os.path.getsize(f) / 1024
    print(f"  - {os.path.basename(f)} ({size_kb:.1f} KB)")


In [ ]:
# Find and load files
def find_file(candidates):
    cand_lower = [c.lower() for c in candidates]
    for c in candidates:
        p = os.path.join(WORKDIR, c)
        if os.path.exists(p):
            return p
    for p in glob.glob(os.path.join(WORKDIR, "*")):
        if os.path.basename(p).lower() in cand_lower:
            return p
    return None

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

PATH_ALT       = find_file(["alternative_roles_analysis.json"])
PATH_JOB_BATCH = find_file(["Job_Classifications_Batch.json"])
PATH_CROSSWALK = find_file(["Role_Confusion_Crosswalk.json"])
PATH_PROMPT    = find_file(["Universal_Role_Classification_Prompt.json"])

print("✅ File Locations:")
print(f"  Alt Roles:  {PATH_ALT}")
print(f"  Job Batch:  {PATH_JOB_BATCH}")
print(f"  Crosswalk:  {PATH_CROSSWALK}")
print(f"  Prompt:     {PATH_PROMPT}")

# Load data
if PATH_ALT:
    alt_data = load_json(PATH_ALT)
if PATH_JOB_BATCH:
    job_data = load_json(PATH_JOB_BATCH)
if PATH_CROSSWALK:
    cross_data = load_json(PATH_CROSSWALK)


---
## 📊 DIAGNOSTIC 1: Alternative Roles File Structure


In [ ]:
print("="*70)
print("ALTERNATIVE ROLES ANALYSIS - FILE STRUCTURE")
print("="*70)

if PATH_ALT:
    print("\n1️⃣ TOP-LEVEL STRUCTURE:")
    print(f"   Type: {type(alt_data)}")
    
    if isinstance(alt_data, dict):
        print(f"   Keys: {list(alt_data.keys())}")
        # Check if wrapped
        if 'rows' in alt_data:
            print("   ✅ Data wrapped in 'rows' key")
            alt_list = alt_data['rows']
        elif 'data' in alt_data:
            print("   ✅ Data wrapped in 'data' key")
            alt_list = alt_data['data']
        else:
            print("   ⚠️ Data NOT wrapped - using dict directly")
            alt_list = [alt_data]
    else:
        print("   ✅ Data is a list")
        alt_list = alt_data
    
    df_alt = pd.DataFrame(alt_list)
    
    print(f"\n2️⃣ DATAFRAME INFO:")
    print(f"   Shape: {df_alt.shape} (rows, columns)")
    print(f"   Columns: {list(df_alt.columns)}")
    
    print(f"\n3️⃣ COLUMN ANALYSIS:")
    
    # Check for title columns
    title_candidates = ['job_title_original', 'job_title', 'job title', 'title', 'job_title_key', 'Job Title']
    found_title_cols = [c for c in df_alt.columns if c in title_candidates or c.lower() in [x.lower() for x in title_candidates]]
    print(f"   Title columns found: {found_title_cols}")
    
    # Check for job code columns
    code_candidates = ['job_code', 'jobcode', 'job code', 'Job Code', 'job_code_number']
    found_code_cols = [c for c in df_alt.columns if c in code_candidates or c.lower().replace(' ','') in [x.lower().replace(' ','') for x in code_candidates]]
    print(f"   Job Code columns found: {found_code_cols}")
    
    # Check for alternative roles columns
    alt_candidates = ['alternative_roles', 'alt_roles', 'alternatives', 'other_roles', 'competing_roles']
    found_alt_cols = [c for c in df_alt.columns if c in alt_candidates or c.lower().replace('_','') in [x.lower().replace('_','') for x in alt_candidates]]
    print(f"   Alternative roles columns found: {found_alt_cols}")
    
    # Check for pattern hit
    pattern_cols = [c for c in df_alt.columns if 'pattern' in c.lower()]
    print(f"   Pattern columns found: {pattern_cols}")
    
    print(f"\n4️⃣ SAMPLE DATA (First 3 rows):")
    display(df_alt.head(3))
    
    print(f"\n5️⃣ DATA TYPES:")
    print(df_alt.dtypes)
    
    # If alternative roles column exists, inspect it
    if found_alt_cols:
        alt_col = found_alt_cols[0]
        print(f"\n6️⃣ ALTERNATIVE ROLES COLUMN DETAILS ('{alt_col}'):")
        print(f"   Sample values:")
        for idx, val in df_alt[alt_col].head(5).items():
            print(f"     Row {idx}: {type(val)} = {val}")
    
else:
    print("❌ Alternative roles file not found!")


---
## 📊 DIAGNOSTIC 2: Job Batch File Structure


In [ ]:
print("="*70)
print("JOB CLASSIFICATIONS BATCH - FILE STRUCTURE")
print("="*70)

if PATH_JOB_BATCH:
    print("\n1️⃣ TOP-LEVEL STRUCTURE:")
    print(f"   Type: {type(job_data)}")
    
    if isinstance(job_data, dict):
        print(f"   Keys: {list(job_data.keys())}")
        if 'rows' in job_data:
            job_list = job_data['rows']
        elif 'data' in job_data:
            job_list = job_data['data']
        else:
            job_list = [job_data]
    else:
        job_list = job_data
    
    df_jobs = pd.DataFrame(job_list)
    
    print(f"\n2️⃣ DATAFRAME INFO:")
    print(f"   Shape: {df_jobs.shape}")
    print(f"   Columns: {list(df_jobs.columns)}")
    
    print(f"\n3️⃣ KEY COLUMNS:")
    
    # Title columns
    title_cols = [c for c in df_jobs.columns if 'title' in c.lower()]
    print(f"   Title-related columns: {title_cols}")
    
    # Role columns
    role_cols = [c for c in df_jobs.columns if 'role' in c.lower() or 'major' in c.lower()]
    print(f"   Role-related columns: {role_cols}")
    
    print(f"\n4️⃣ SAMPLE DATA (First 3 rows):")
    display(df_jobs.head(3))
    
    # Show title values specifically
    if title_cols:
        print(f"\n5️⃣ SAMPLE TITLE VALUES (first 10):")
        for col in title_cols[:2]:  # Show first 2 title columns
            print(f"\n   Column: {col}")
            print("   Values:")
            for val in df_jobs[col].head(10):
                print(f"     '{val}'")
else:
    print("❌ Job batch file not found!")


---
## 📊 DIAGNOSTIC 3: Crosswalk File Structure


In [ ]:
print("="*70)
print("ROLE CONFUSION CROSSWALK - FILE STRUCTURE")
print("="*70)

if PATH_CROSSWALK:
    print("\n1️⃣ TOP-LEVEL STRUCTURE:")
    print(f"   Type: {type(cross_data)}")
    
    if isinstance(cross_data, dict):
        print(f"   Keys: {list(cross_data.keys())}")
        if 'rows' in cross_data:
            cross_list = cross_data['rows']
        elif 'data' in cross_data:
            cross_list = cross_data['data']
        else:
            cross_list = [cross_data]
    else:
        cross_list = cross_data
    
    df_cross = pd.DataFrame(cross_list)
    
    print(f"\n2️⃣ DATAFRAME INFO:")
    print(f"   Shape: {df_cross.shape}")
    print(f"   Columns: {list(df_cross.columns)}")
    
    # Check for confusion/risk columns
    print(f"\n3️⃣ KEY COLUMNS:")
    confusion_cols = [c for c in df_cross.columns if 'confusion' in c.lower() or 'risk' in c.lower()]
    print(f"   Confusion/Risk columns: {confusion_cols}")
    
    role_cols = [c for c in df_cross.columns if 'role' in c.lower()]
    print(f"   Role columns: {role_cols}")
    
    print(f"\n4️⃣ SAMPLE DATA (First 3 rows):")
    display(df_cross.head(3))
    
else:
    print("❌ Crosswalk file not found!")


---
## 🔗 DIAGNOSTIC 4: Merge Compatibility Test


In [ ]:
print("="*70)
print("MERGE COMPATIBILITY ANALYSIS")
print("="*70)

def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return str(s).strip()

if PATH_ALT and PATH_JOB_BATCH:
    print("\n1️⃣ CREATING NORMALIZED KEYS:\n")
    
    # Job batch - find title column
    job_title_col = None
    for col in df_jobs.columns:
        if col.lower() in ['job_title_original', 'job title', 'job_title', 'title', 'new_job_title']:
            job_title_col = col
            break
    
    if job_title_col:
        df_jobs['job_title_key'] = df_jobs[job_title_col].map(norm)
        print(f"   Job Batch using column: '{job_title_col}'")
        print(f"   Sample keys:")
        for key in df_jobs['job_title_key'].head(10):
            print(f"     '{key}'")
    else:
        print("   ❌ No title column found in job batch!")
    
    print("\n")
    
    # Alt analysis - find title or job code
    alt_key_col = None
    
    # Try job code first
    for col in df_alt.columns:
        if col.lower().replace(' ', '') in ['jobcode', 'job_code', 'jobcodenumber']:
            alt_key_col = col
            print(f"   Alt Roles using JOB CODE column: '{alt_key_col}'")
            break
    
    # Fall back to title
    if not alt_key_col:
        for col in df_alt.columns:
            if col.lower() in ['job_title_original', 'job title', 'job_title', 'title', 'job_title_key']:
                alt_key_col = col
                print(f"   Alt Roles using TITLE column: '{alt_key_col}'")
                break
    
    if alt_key_col:
        df_alt['job_title_key'] = df_alt[alt_key_col].map(norm)
        print(f"   Sample keys:")
        for key in df_alt['job_title_key'].head(10):
            print(f"     '{key}'")
    else:
        print("   ❌ No suitable key column found in alt roles!")
    
    print("\n2️⃣ MERGE TEST:\n")
    
    if job_title_col and alt_key_col:
        # Find matching keys
        job_keys = set(df_jobs['job_title_key'].unique())
        alt_keys = set(df_alt['job_title_key'].unique())
        
        matching_keys = job_keys & alt_keys
        only_in_jobs = job_keys - alt_keys
        only_in_alt = alt_keys - job_keys
        
        print(f"   Total job batch keys:      {len(job_keys)}")
        print(f"   Total alt roles keys:      {len(alt_keys)}")
        print(f"   Matching keys:             {len(matching_keys)} ✅")
        print(f"   Only in job batch:         {len(only_in_jobs)}")
        print(f"   Only in alt roles:         {len(only_in_alt)}")
        
        print(f"\n   Match rate: {len(matching_keys)/len(job_keys)*100:.1f}%")
        
        if len(matching_keys) == 0:
            print("\n   ❌ ZERO MATCHES FOUND!")
            print("\n   Comparing sample keys:")
            print("\n   First 5 from Job Batch:")
            for k in list(job_keys)[:5]:
                print(f"     '{k}'")
            print("\n   First 5 from Alt Roles:")
            for k in list(alt_keys)[:5]:
                print(f"     '{k}'")
                
            # Check for case sensitivity issues
            job_keys_lower = set(k.lower() for k in job_keys)
            alt_keys_lower = set(k.lower() for k in alt_keys)
            case_insensitive_matches = job_keys_lower & alt_keys_lower
            
            if len(case_insensitive_matches) > 0:
                print(f"\n   ⚠️ {len(case_insensitive_matches)} case-insensitive matches found!")
                print("   Issue: Keys differ by case/formatting")
        
        elif len(matching_keys) < len(job_keys) * 0.5:
            print("\n   ⚠️ LOW MATCH RATE - some keys not matching")
            print("\n   Sample non-matching job keys:")
            for k in list(only_in_jobs)[:5]:
                print(f"     '{k}'")
        else:
            print("\n   ✅ Good match rate!")
            print("\n   Sample matching keys:")
            for k in list(matching_keys)[:5]:
                print(f"     '{k}'")
    
else:
    print("❌ Cannot perform merge test - missing files")


---
## 📋 DIAGNOSTIC 5: Summary & Recommendations


In [ ]:
print("="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70)

print("\n🔍 KEY FINDINGS:\n")

issues = []
recommendations = []

# Check alt roles structure
if PATH_ALT and 'df_alt' in locals():
    if df_alt.shape[0] == 0:
        issues.append("Alternative roles file is EMPTY")
        recommendations.append("Verify alternative_roles_analysis.json has data")
    else:
        print(f"✅ Alternative roles file loaded: {df_alt.shape[0]} records")
        
        # Check for alt roles column
        if not found_alt_cols:
            issues.append("No alternative roles column found in alt file")
            recommendations.append(f"Expected columns like: {alt_candidates}")
            recommendations.append(f"Actual columns: {list(df_alt.columns)}")
else:
    issues.append("Alternative roles file NOT loaded")
    recommendations.append("Ensure alternative_roles_analysis.json is uploaded")

# Check job batch
if PATH_JOB_BATCH and 'df_jobs' in locals():
    print(f"✅ Job batch loaded: {df_jobs.shape[0]} records")
else:
    issues.append("Job batch file NOT loaded")

# Check merge compatibility
if 'matching_keys' in locals():
    if len(matching_keys) == 0:
        issues.append("ZERO key matches between files")
        recommendations.append("Key columns are using different values/formats")
        recommendations.append("Need custom merge logic or key transformation")
    elif len(matching_keys) < len(job_keys) * 0.5:
        issues.append(f"Only {len(matching_keys)}/{len(job_keys)} keys match ({len(matching_keys)/len(job_keys)*100:.1f}%)")
        recommendations.append("Consider fuzzy matching or key normalization")
    else:
        print(f"✅ Good key matching: {len(matching_keys)}/{len(job_keys)} ({len(matching_keys)/len(job_keys)*100:.1f}%)")

if issues:
    print("\n❌ ISSUES FOUND:\n")
    for i, issue in enumerate(issues, 1):
        print(f"   {i}. {issue}")
    
    print("\n💡 RECOMMENDATIONS:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"   {i}. {rec}")
else:
    print("\n✅ No critical issues found!")
    print("\n   Data structure looks compatible.")
    print("   Issue likely in notebook merge logic.")

print("\n" + "="*70)
print("Share these results to get a customized fix!")
print("="*70)
